# Controlled Descent Simulator — Quadrotor Model for Nonlinear MPC (iLQR)

This notebook derives the **prediction model** used by a nonlinear Model-Predictive
Controller for the quadrotor. It is the MPC counterpart of the FF+LQR derivation
in `dynamics_quadRotor_FFLQR01.ipynb`, and it deliberately reuses the *same*
6-DOF plant so the two controllers are compared on equal footing.

**What an MPC needs from a model.** An MPC does not use a fixed control law.
At every control instant it *predicts* how the vehicle will move over a short
horizon of `N` steps under a candidate sequence of motor commands, scores those
predictions against a reference, and picks the sequence that minimises the score
subject to the actuator limits — then applies only the first command and repeats.
So the model has to be a **prediction machine**:

$$x_{k+1} = F_d(x_k,\,u_k),\qquad k = 0,\dots,N-1,$$

together with its **sensitivities**

$$A_k = \frac{\partial F_d}{\partial x}\Big|_{x_k,u_k},\qquad
  B_k = \frac{\partial F_d}{\partial u}\Big|_{x_k,u_k}.$$

**Why the Jacobians.** The solver we chose is **iterative LQR (iLQR)** — the control-limited form of
**DDP, Differential Dynamic Programming** (Jacobson & Mayne, 1970; Li & Todorov,
2004; Tassa, Erez & Todorov, 2014). It is exactly what its name says: it linearises the nonlinear rollout around
the current guess and solves an LQR-like sub-problem (a backward Riccati-style
recursion) to improve it, iterating to convergence. That backward pass consumes
`A_k` and `B_k` at every knot. This is the one structural difference from the
LQR notebook: LQR linearises **once**, at hover, and ships a constant gain;
MPC re-linearises the **actual predicted trajectory** on every tick.

**This step builds only the model** — the discrete map `F_d` and its Jacobians,
validated numerically. The cost function, the horizon, and the backward/forward
passes come in the following steps.

In [ ]:
import sympy as sp
from sympy import symbols, Matrix, sqrt, Rational, diag
sp.init_printing(use_latex='mathjax', wrap_line=False)

import numpy as np

print("SymPy:", sp.__version__)
print("NumPy:", np.__version__)

## 1. Continuous nonlinear 6-DOF model

We take the **same plant** as the LQR derivation: a rigid body with a unit
quaternion for attitude, world-frame translational dynamics, body-frame Euler
rotational dynamics, and a body-attached linear drag. The state is 13-dimensional

$$x = [\;r_{x,y,z},\; q_{w,x,y,z},\; v_{x,y,z},\; \omega_{x,y,z}\;]^\top .$$

**The control is the four motor thrusts** $u = [T_1,T_2,T_3,T_4]^\top$, *not* the
virtual wrench $[F,\tau]$. This is a deliberate MPC modelling choice: the physical
limits of the vehicle are per-rotor thrust bounds $T_i\in[T_\mathrm{min},T_\mathrm{max}]$, and by
optimising directly over $T_i$ those limits become a simple **box constraint** —
which control-limited DDP handles natively in its backward pass. (LQR did the
opposite: it computed a wrench, then inverted the allocation matrix and clamped.)

In [ ]:
# ---- symbolic parameters (same airframe as the LQR notebook) ----
m, Ixx, Iyy, Izz, g = symbols('m I_xx I_yy I_zz g', positive=True)
d_x, d_y, d_z       = symbols('d_x d_y d_z', positive=True)      # world/body linear-drag diag
k_T, k_Q, I_rot     = symbols('k_T k_Q I_rot', positive=True)    # thrust/torque coeffs, rotor inertia
L                   = symbols('L', positive=True)                # arm length (centre -> motor)

# ---- state variables ----
r_x, r_y, r_z       = symbols('r_x r_y r_z', real=True)
q_w, q_x, q_y, q_z  = symbols('q_w q_x q_y q_z', real=True)
v_x, v_y, v_z       = symbols('v_x v_y v_z', real=True)
w_x, w_y, w_z       = symbols('w_x w_y w_z', real=True)          # body rates omega

# ---- control inputs: the four motor thrusts ----
T1, T2, T3, T4      = symbols('T1 T2 T3 T4', real=True)

# ---- external perturbation force (world frame): a plant input, zero for the predictor ----
uFx, uFy, uFz       = symbols('uFx uFy uFz', real=True)

q_vec = Matrix([q_w, q_x, q_y, q_z])
w_vec = Matrix([w_x, w_y, w_z])
v_vec = Matrix([v_x, v_y, v_z])
Imat  = diag(Ixx, Iyy, Izz)

### 1.1 Rotation matrix and attitude kinematics

The body-to-world rotation from the quaternion, and the quaternion derivative
$\dot q = \tfrac12\,\Omega(\omega)\,q$. The unit-norm constraint $\|q\|=1$ is a
manifold that ordinary integration does not preserve exactly — we deal with it in
§3 by renormalising after each step.

In [ ]:
def R_of_q(qw, qx, qy, qz):
    return Matrix([
        [1-2*(qy**2+qz**2), 2*(qx*qy-qw*qz),   2*(qx*qz+qw*qy)],
        [2*(qx*qy+qw*qz),   1-2*(qx**2+qz**2), 2*(qy*qz-qw*qx)],
        [2*(qx*qz-qw*qy),   2*(qy*qz+qw*qx),   1-2*(qx**2+qy**2)],
    ])

R = R_of_q(q_w, q_x, q_y, q_z)

Omega = Matrix([
    [0,   -w_x, -w_y, -w_z],
    [w_x,  0,    w_z, -w_y],
    [w_y, -w_z,  0,    w_x],
    [w_z,  w_y, -w_x,  0  ],
])
q_dot = Rational(1, 2) * Omega * q_vec
q_dot

### 1.2 Control allocation — from motor thrusts to the wrench

The four thrusts produce a collective force $F$ and a body torque $\tau$ through
the fixed **ArduPilot QuadX** allocation matrix $M$ (identical to §2 of the LQR
notebook). Because $u = T$, the wrench is simply the affine map $[F,\tau]^\top =
M\,T$, and we substitute it into the dynamics — the allocation lives *inside* the
model instead of being inverted afterwards.

In [ ]:
d   = L / sqrt(2)          # motor x,y offset on the 45-degree X
c_r = k_Q / k_T            # torque-to-thrust ratio

#              M1(FR)   M2(RL)   M3(FL)   M4(RR)
pos = [( d, -d), (-d,  d), ( d,  d), (-d, -d)]
sig = [   +1,      +1,      -1,      -1    ]      # +1 = CCW, -1 = CW

M_alloc = Matrix([
    [1, 1, 1, 1],
    [ pos[0][1],  pos[1][1],  pos[2][1],  pos[3][1]],      # tau_x =  sum T*y
    [-pos[0][0], -pos[1][0], -pos[2][0], -pos[3][0]],      # tau_y = -sum T*x
    [-sig[0]*c_r, -sig[1]*c_r, -sig[2]*c_r, -sig[3]*c_r],  # tau_z =  sum -sigma*c*T
])
wrench = M_alloc * Matrix([T1, T2, T3, T4])
F, tau_x, tau_y, tau_z = wrench
M_alloc

### 1.3 Translational and rotational dynamics

Translational (world frame): thrust rotated into the world, gravity, and a
body-attached linear drag $-R\,D\,R^\top v$. Rotational (body frame): the Euler
equation $I\dot\omega = \tau - \omega\times I\omega$.

**One deliberate simplification vs the plant:** we drop the residual rotor
gyroscopic term $\omega\times[0,0,I_\text{rot}S]^\top$. At a symmetric hover the
signed spin sum $S=\sum_i \sigma_i\sqrt{T_i/k_T}$ is $\approx 0$, and — crucially
for MPC — its $\sqrt{T_i}$ dependence has an **unbounded derivative at
$T_i=0=T_\mathrm{min}$**, which would make $B_k$ singular exactly on the constraint
boundary. Neglecting a tiny, non-smooth term buys well-conditioned Jacobians
across the whole feasible box. It can be reinstated later as a smoothed term if
its effect proves non-negligible.

The translational equation also carries an **external world-frame force** $u_F$ (wind, contacts): a *plant* input, exercised in the sim and set to zero in the predictor the solver rolls out. Being additive, it leaves $f_x,f_u$ untouched.

In [ ]:
D_mat        = diag(d_x, d_y, d_z)
thrust_world = R * Matrix([0, 0, F])
gravity      = Matrix([0, 0, -m*g])
v_body       = R.T * v_vec
drag_world   = R * (-D_mat * v_body)
v_dot        = (thrust_world + gravity + drag_world + Matrix([uFx, uFy, uFz])) / m

tau_vec = Matrix([tau_x, tau_y, tau_z])
w_dot   = Imat.inv() * (tau_vec - w_vec.cross(Imat * w_vec))   # rotor gyro neglected (see above)
v_dot, w_dot

### 1.4 Assembled continuous model $\dot x = f(x,u)$

In [ ]:
state = Matrix([r_x, r_y, r_z,
                q_w, q_x, q_y, q_z,
                v_x, v_y, v_z,
                w_x, w_y, w_z])
ctrl  = Matrix([T1, T2, T3, T4])

f = Matrix([v_x, v_y, v_z,      # r_dot = v
            *q_dot,             # q_dot = 1/2 Omega(omega) q
            *v_dot,             # v_dot (translational)
            *w_dot])            # omega_dot (rotational)

print("state dim:", state.shape[0], " | input dim:", ctrl.shape[0], " | f dim:", f.shape[0])

### 1.5 Sanity check — hover is an equilibrium

With every motor at $T_i = mg/4$ the wrench is $[mg,0,0,0]$ and $f(x,u)=0$ at the
level, motionless state. This is the point the MPC will hold, and the natural **warm start** — the initial
guess the iterative solver of §5 begins from — for the very first solve.

In [ ]:
params = {                       # ~2.4 kg / 550 mm airframe
    m: 2.4, Ixx: 0.025, Iyy: 0.025, Izz: 0.045, g: 9.81,
    d_x: 0.20, d_y: 0.20, d_z: 0.30,
    k_T: 1.0e-5, k_Q: 1.6e-7, I_rot: 3.0e-5, L: 0.275,
}
m_val, g_val = params[m], params[g]

args   = (r_x, r_y, r_z, q_w, q_x, q_y, q_z, v_x, v_y, v_z, w_x, w_y, w_z, T1, T2, T3, T4)  # Jacobian args
args_f = args + (uFx, uFy, uFz)                        # + world-frame external force (plant map)
f_l = sp.lambdify(args_f, f.subs(params), 'numpy')
def f_num(x, u, uF=(0.0, 0.0, 0.0)):  return np.asarray(f_l(*x, *u, *uF), float).flatten()

T_hover = m_val * g_val / 4.0
x_hover = np.array([0,0,0, 1,0,0,0, 0,0,0, 0,0,0], float)
u_hover = np.array([T_hover]*4, float)
print("per-rotor hover thrust [N]:", round(T_hover, 4))
print("f at hover:", np.round(f_num(x_hover, u_hover), 12))
assert np.allclose(f_num(x_hover, u_hover), 0, atol=1e-9), "hover is not an equilibrium!"
print("OK - hover is an equilibrium.")

## 2. Discretization — the prediction map $F_d$

The MPC reasons on a discrete grid $k=0,1,\dots,N$ spaced by the control period
$\Delta t_\text{mpc}$, holding each $u_k$ constant over its interval (**zero-order hold**). We integrate
$f$ over one interval with classic **RK4**, which is accurate enough at the
tens-of-milliseconds control period while staying cheap to differentiate:

$$
k_1=f(x,u),\quad k_2=f(x+\tfrac{\Delta t}{2}k_1,u),\quad
k_3=f(x+\tfrac{\Delta t}{2}k_2,u),\quad k_4=f(x+\Delta t\,k_3,u),
$$
$$
F_d(x,u) = x + \tfrac{\Delta t}{6}\,(k_1+2k_2+2k_3+k_4).
$$

**Quaternion renormalisation.** RK4 does not preserve $\|q\|=1$ exactly, so after
each step we project the quaternion block back onto the unit sphere (divide it by
its norm). For the Jacobian derivation in §3 we
keep the *pure* RK4 map (renorm treated as a separate projection, whose Jacobian
is stated there), which keeps the acid test unambiguous.

In [ ]:
DT_MPC = 0.02   # control period [s]  (50 Hz) -- an MPC knob, tuned in a later step

def rk4_step(x, u, dt=DT_MPC):
    k1 = f_num(x, u)
    k2 = f_num(x + 0.5*dt*k1, u)
    k3 = f_num(x + 0.5*dt*k2, u)
    k4 = f_num(x + dt*k3, u)
    return x + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def normalize_quat(x):
    y = x.copy()
    y[3:7] /= np.linalg.norm(y[3:7])
    return y

def F_d(x, u, dt=DT_MPC):        # discrete prediction map used at runtime (with renorm)
    return normalize_quat(rk4_step(x, u, dt))

# one predicted step from hover with a small roll asymmetry
u_test = u_hover + np.array([0.5, -0.5, 0.0, 0.0])
print("x_{k+1} - x_k =", np.round(F_d(x_hover, u_test) - x_hover, 5))

## 3. Jacobians of the prediction — $A_k$ and $B_k$

The solver of §5 works with a *linear* model of how the discrete step $F_d$ reacts
to small changes in the state and the command:

$$A_k = \frac{\partial F_d}{\partial x}\Big|_{x_k,u_k}\ (13\times13),\qquad
  B_k = \frac{\partial F_d}{\partial u}\Big|_{x_k,u_k}\ (13\times4).$$

We already have the Jacobians of the *continuous* $f$ — $f_x=\partial f/\partial x$
and $f_u=\partial f/\partial u$ (built symbolically below). The question is how to
get the Jacobians of the *discrete* $F_d$, which is $f$ wrapped in four RK4 stages.
Two routes give the same answer:

1. **Symbolic Jacobian of $F_d$** — differentiate the entire RK4 expression at once.
   Exact, but the expression tree is huge and slow.
2. **Sensitivity propagation (what we use)** — recall that $F_d$ is a *composition*:
   stage $k_2$ is evaluated at a point that already depends on $k_1$, $k_3$ on $k_2$,
   and so on. Differentiating a composition is the **chain rule**. So we carry the
   derivative *along with* the value through the four stages: at each stage we
   evaluate $f_x,f_u$ there and multiply by the sensitivity accumulated so far.
   No giant expression, and it needs only $f,f_x,f_u$.

Writing $s_i=\partial k_i/\partial x$ (and analogously for $u$), the chain rule
through the stages gives

$$
s_1 = f_x(x),\quad
s_2 = f_x\!\big(x{+}\tfrac{\Delta t}{2}k_1\big)\big(I{+}\tfrac{\Delta t}{2}s_1\big),\quad
s_3 = f_x\!\big(x{+}\tfrac{\Delta t}{2}k_2\big)\big(I{+}\tfrac{\Delta t}{2}s_2\big),\quad
s_4 = f_x\!\big(x{+}\Delta t\,k_3\big)\big(I{+}\Delta t\,s_3\big),
$$

and the discrete Jacobian is the **same weighted average** that RK4 uses for the
state itself, $A_k = I + \tfrac{\Delta t}{6}(s_1+2s_2+2s_3+s_4)$ (and $B_k$ the same
with the $f_u$ contributions). The $I$ appears because $F_d$ starts from $x$ itself.

<div align="center"><svg viewBox="0 0 720 250" xmlns="http://www.w3.org/2000/svg" font-family="system-ui,sans-serif" font-size="12.5">
  <rect x="1" y="1" width="718" height="248" rx="10" fill="#ffffff" stroke="#e2e8f0"/>
  <defs><marker id="ah3" markerWidth="9" markerHeight="9" refX="6" refY="3" orient="auto"><path d="M0,0 L6,3 L0,6 Z" fill="#334155"/></marker></defs>
  <text x="360" y="26" text-anchor="middle" font-weight="600" fill="#0f172a">one RK4 step = four chained stage evaluations of f</text>
  <g fill="#0f172a">
    <rect x="30"  y="46" width="120" height="34" rx="5" fill="#dbeafe" stroke="#1e40af"/><text x="90"  y="68" text-anchor="middle">k&#8321; = f(x)</text>
    <rect x="200" y="46" width="120" height="34" rx="5" fill="#dbeafe" stroke="#1e40af"/><text x="260" y="68" text-anchor="middle">k&#8322; = f(x+&#189;&#916;t k&#8321;)</text>
    <rect x="370" y="46" width="120" height="34" rx="5" fill="#dbeafe" stroke="#1e40af"/><text x="430" y="68" text-anchor="middle">k&#8323; = f(x+&#189;&#916;t k&#8322;)</text>
    <rect x="540" y="46" width="120" height="34" rx="5" fill="#dbeafe" stroke="#1e40af"/><text x="600" y="68" text-anchor="middle">k&#8324; = f(x+&#916;t k&#8323;)</text>
  </g>
  <line x1="152" y1="63" x2="198" y2="63" stroke="#334155" stroke-width="1.5" marker-end="url(#ah3)"/>
  <line x1="322" y1="63" x2="368" y2="63" stroke="#334155" stroke-width="1.5" marker-end="url(#ah3)"/>
  <line x1="492" y1="63" x2="538" y2="63" stroke="#334155" stroke-width="1.5" marker-end="url(#ah3)"/>
  <text x="360" y="110" text-anchor="middle" fill="#b45309" font-size="12.5">differentiate each stage by the chain rule, reusing f&#8339;, f&#7512; of that stage:</text>
  <g fill="#0f172a" font-size="12">
    <text x="90"  y="140" text-anchor="middle">s&#8321; = f&#8339;(x)</text>
    <text x="260" y="140" text-anchor="middle">s&#8322; = f&#8339;&#183;(I+&#189;&#916;t s&#8321;)</text>
    <text x="430" y="140" text-anchor="middle">s&#8323; = f&#8339;&#183;(I+&#189;&#916;t s&#8322;)</text>
    <text x="600" y="140" text-anchor="middle">s&#8324; = f&#8339;&#183;(I+&#916;t s&#8323;)</text>
  </g>
  <text x="360" y="140" text-anchor="middle" fill="#94a3b8">&#8594;</text>
  <line x1="360" y1="150" x2="360" y2="176" stroke="#334155" stroke-width="1.6" marker-end="url(#ah3)"/>
  <rect x="150" y="180" width="420" height="46" rx="7" fill="#f8fafc" stroke="#64748b"/>
  <text x="360" y="200" text-anchor="middle" fill="#0f172a" font-size="14">A&#8342; = I + (&#916;t/6)(s&#8321;+2s&#8322;+2s&#8323;+s&#8324;)   ,   B&#8342; likewise with f&#7512;</text>
  <text x="360" y="219" text-anchor="middle" fill="#15803d" font-size="11.5">the discrete Jacobians &#8212; same weights (1,2,2,1) as the RK4 sum itself</text>
</svg></div>

The quaternion renormalisation is a separate projection onto $\|q\|=1$; its
Jacobian is $P = (I - qq^\top/\|q\|^2)/\|q\|$ on the quaternion block, which near
$\|q\|=1$ is $\approx I - qq^\top$. To keep the check in §3.1 unambiguous we
derive $A_k,B_k$ for the **pure** RK4 map (no renorm) and verify *that*.

In [ ]:
fx = f.jacobian(state)
fu = f.jacobian(ctrl)
fx_l = sp.lambdify(args, fx.subs(params), 'numpy')
fu_l = sp.lambdify(args, fu.subs(params), 'numpy')
def fx_num(x, u): return np.asarray(fx_l(*x, *u), float)
def fu_num(x, u): return np.asarray(fu_l(*x, *u), float)
print("f_x shape:", fx.shape, " f_u shape:", fu.shape)

In [ ]:
def rk4_jacobians(x, u, dt=DT_MPC):
    I = np.eye(len(x))
    # stage states (same as rk4_step) ...
    k1 = f_num(x, u)
    x2 = x + 0.5*dt*k1;   k2 = f_num(x2, u)
    x3 = x + 0.5*dt*k2;   k3 = f_num(x3, u)
    x4 = x + dt*k3
    # ... and their sensitivities by the chain rule
    A1 = fx_num(x,  u);                     B1 = fu_num(x,  u)
    J2 = fx_num(x2, u)
    A2 = J2 @ (I + 0.5*dt*A1);              B2 = J2 @ (0.5*dt*B1) + fu_num(x2, u)
    J3 = fx_num(x3, u)
    A3 = J3 @ (I + 0.5*dt*A2);              B3 = J3 @ (0.5*dt*B2) + fu_num(x3, u)
    J4 = fx_num(x4, u)
    A4 = J4 @ (I + dt*A3);                  B4 = J4 @ (dt*B3)     + fu_num(x4, u)
    A = I + (dt/6.0)*(A1 + 2*A2 + 2*A3 + A4)
    B =     (dt/6.0)*(B1 + 2*B2 + 2*B3 + B4)
    return A, B

### 3.1 Acid test — analytic Jacobians vs finite differences, at many points

The propagated $A_k,B_k$ must reproduce a central **finite difference** of the pure
RK4 map (no renorm on either side). One lucky point could hide a bug, so we check
the worst error over **several random operating points**.

In [ ]:
rng = np.random.default_rng(0)
eps = 1e-6
worst_A = worst_B = 0.0
for _ in range(10):
    x_p = normalize_quat(x_hover + rng.normal(0, 0.15, 13))
    u_p = u_hover + rng.normal(0, 0.6, 4)
    A, B = rk4_jacobians(x_p, u_p)
    A_fd = np.stack([(rk4_step(x_p+e, u_p) - rk4_step(x_p-e, u_p))/(2*eps) for e in eps*np.eye(13)], axis=1)
    B_fd = np.stack([(rk4_step(x_p, u_p+e) - rk4_step(x_p, u_p-e))/(2*eps) for e in eps*np.eye(4)],  axis=1)
    worst_A = max(worst_A, np.max(np.abs(A - A_fd)))
    worst_B = max(worst_B, np.max(np.abs(B - B_fd)))
print(f"over 10 random points:  max|A - A_fd| = {worst_A:.2e}   max|B - B_fd| = {worst_B:.2e}")
assert worst_A < 1e-6 and worst_B < 1e-6
print("ACID TEST PASSED - the prediction Jacobians are correct at every point.")

## 4. The cost function

The predictor of §2–§3 tells us *where a candidate command sequence takes the
vehicle*. To choose among sequences we need a number that says *how good* a
predicted trajectory is. That number is the **cost**, and the MPC minimises it
over the horizon:

$$J(x_0,\mathbf{u}) \;=\; \underbrace{\sum_{k=0}^{N-1}\ell(x_k,u_k)}_{\text{running (stage) cost}} \;+\; \underbrace{\ell_f(x_N)}_{\text{terminal cost}},
\qquad \mathbf{u}=(u_0,\dots,u_{N-1}).$$

Each **stage cost** $\ell(x_k,u_k)$ charges the trajectory for two things at step
$k$: being *off the reference* (tracking error) and *spending control effort*.
The **terminal cost** $\ell_f(x_N)$ charges the final state; §4.4 explains why a
finite horizon needs it. This quadratic-tracking form is the standard MPC
objective (Rawlings, Mayne & Diehl, *Model Predictive Control*, 2nd ed., 2017, §2).

The rest of §4 builds $\ell$ and $\ell_f$ **and** the derivatives the solver of §5
will consume — in an order chosen so nothing is used before it is defined.

### 4.1 What we track — the reference

At every step the cost compares the state $x_k$ against a **reference** $x^{\text{ref}}_k$:
the state we *want* the vehicle to be in at that instant. It has one entry per
physical quantity in the state:

| symbol | meaning | tracked against |
|---|---|---|
| $p^{\text{ref}}=(p_{rx},p_{ry},p_{rz})$ | desired **position** [m] | $(r_x,r_y,r_z)$ |
| $q^{\text{ref}}=(q_{rw},q_{rx},q_{ry},q_{rz})$ | desired **attitude** (unit quaternion) | $(q_w,q_x,q_y,q_z)$ |
| $v^{\text{ref}}=(v_{rx},v_{ry},v_{rz})$ | desired **velocity** [m/s] | $(v_x,v_y,v_z)$ |
| $w^{\text{ref}}=(w_{rx},w_{ry},w_{rz})$ | desired **body rates** [rad/s] | $(\omega_x,\omega_y,\omega_z)$ |
| $u^{\text{ref}}=(u_{r1},\dots,u_{r4})$ | **nominal command** (per-rotor thrust) | $(T_1,\dots,T_4)$ |

A whole *reference trajectory* is just this vector at each step $k$; a plain
hover set-point is the special case where it is constant. The control reference
$u^{\text{ref}}$ is the hover thrust $mg/4$ — the effort penalty pulls the
commands towards *hovering*, not towards *zero* (which would be free-fall).

### 4.2 Writing the cost as squared residuals — the Gauss–Newton idea

We could hand the solver any smooth $\ell$. We deliberately write it as **one
half of a squared vector of residuals**,

$$\ell(x,u)=\tfrac12\,\lVert r(x,u)\rVert^2,\qquad r\in\mathbb{R}^{m},$$

where each entry of $r$ is one weighted error (a position error, an attitude
error, …). This is not cosmetic. Two quantities the backward pass of §5 needs are
the **gradient** and the **Hessian** of $\ell$ — *by definition*

$$\ell_x \;\equiv\; \frac{\partial \ell}{\partial x}\ \ (\text{a vector}),
\qquad
\ell_{xx} \;\equiv\; \frac{\partial^2 \ell}{\partial x^2}\ \ (\text{a matrix}).$$

These are **derivatives *of* $\ell$**; they do not appear inside the expression
for $\ell$. For the squared-residual form the chain rule gives them in closed
form. With $J=\partial r/\partial x$ (the Jacobian of the residual),

$$\ell_x = J^\top r,\qquad
\ell_{xx} = \underbrace{J^\top J}_{\text{Gauss–Newton}} + \sum_i r_i\,\frac{\partial^2 r_i}{\partial x^2}.$$

The **Gauss–Newton approximation** (Nocedal & Wright, *Numerical Optimization*,
2nd ed., 2006, §10.3) drops the second term — small near the reference and
possibly indefinite — and keeps $\ell_{xx}\approx J^\top J$. That single choice
buys the property the solver relies on: $J^\top J$ is **positive semidefinite for
free**, so every quadratic sub-problem in §5 is convex without any patching. This
is the "Gauss–Newton flavour" of iLQR.

<div align="center"><svg viewBox="0 0 720 250" xmlns="http://www.w3.org/2000/svg" font-family="system-ui,sans-serif" font-size="12.5">
  <rect x="1" y="1" width="718" height="248" rx="10" fill="#ffffff" stroke="#e2e8f0"/>
  <defs><marker id="ah4a" markerWidth="9" markerHeight="9" refX="6" refY="3" orient="auto"><path d="M0,0 L6,3 L0,6 Z" fill="#334155"/></marker></defs>
  <text x="97" y="26" text-anchor="middle" font-weight="600" fill="#0f172a">residual  r(x,u)</text>
  <g fill="#0f172a">
    <rect x="38" y="38" width="118" height="27" fill="#dbeafe" stroke="#1e40af"/><text x="97" y="56" text-anchor="middle">position &#183; 3</text>
    <rect x="38" y="65" width="118" height="27" fill="#fde68a" stroke="#b45309"/><text x="97" y="83" text-anchor="middle">attitude &#183; 3</text>
    <rect x="38" y="92" width="118" height="27" fill="#dbeafe" stroke="#1e40af"/><text x="97" y="110" text-anchor="middle">velocity &#183; 3</text>
    <rect x="38" y="119" width="118" height="27" fill="#dbeafe" stroke="#1e40af"/><text x="97" y="137" text-anchor="middle">body rate &#183; 3</text>
    <rect x="38" y="146" width="118" height="27" fill="#dcfce7" stroke="#15803d"/><text x="97" y="164" text-anchor="middle">control &#183; 4</text>
  </g>
  <text x="97" y="190" text-anchor="middle" fill="#64748b" font-size="11.5">weighted errors, stacked</text>
  <line x1="160" y1="105" x2="214" y2="105" stroke="#334155" stroke-width="1.6" marker-end="url(#ah4a)"/>
  <text x="330" y="74" text-anchor="middle" font-size="19" fill="#0f172a">&#8467; = &#189; &#8214;r&#8214;&#178;</text>
  <text x="330" y="94" text-anchor="middle" fill="#64748b" font-size="11.5">one scalar score</text>
  <line x1="330" y1="106" x2="330" y2="130" stroke="#334155" stroke-width="1.6" marker-end="url(#ah4a)"/>
  <rect x="212" y="134" width="236" height="46" rx="7" fill="#f8fafc" stroke="#64748b"/>
  <text x="330" y="155" text-anchor="middle" font-size="15" fill="#0f172a">&#8711;&#8467; = J&#7488;r      &#8711;&#178;&#8467; &#8776; J&#7488;J</text>
  <text x="330" y="173" text-anchor="middle" fill="#64748b" font-size="11">J = &#8706;r/&#8706;x  (chain rule)</text>
  <text x="330" y="200" text-anchor="middle" fill="#15803d" font-size="12.5">J&#7488;J is PSD &#8212; a convex sub-problem, for free</text>
  <text x="600" y="150" text-anchor="middle" fill="#64748b" font-size="11.5">the solver of &#167;5</text>
  <text x="600" y="166" text-anchor="middle" fill="#64748b" font-size="11.5">reads these two</text>
</svg></div>

### 4.3 The residual vector, and why attitude is special

The residual stacks one weighted block per physical error. Position, velocity and
body-rate errors are ordinary vector differences (state minus reference). The
**attitude** block is not: a unit quaternion lives on a sphere, so
$q-q^{\text{ref}}$ is meaningless as an error — it ignores that $q$ and $-q$ are
the *same* rotation (the double cover) and that a 4-vector difference is not a
distance on the rotation group $SO(3)$.

The correct small error is the **error quaternion**
$q_e = q_{\text{ref}}^{*}\otimes q$ — the rotation that carries the reference
attitude onto the current one ($q^{*}$ is the conjugate/inverse, $\otimes$ the
quaternion product). Its vector part, doubled, is the rotation-vector error
$\delta\theta = 2\,\mathrm{vec}(q_e)$: for a small misalignment $\delta\theta$ is
the axis-times-angle you would rotate through to fix it (Solà, *Quaternion
kinematics for the error-state Kalman filter*, 2017, §4.5). It is the same
reduction the FF+LQR notebook used, and it stays valid well short of the
$180^{\circ}$ point where the shortest-rotation sign flips.

**A quaternion carries 4 numbers but only 3 rotational degrees of freedom.** The
attitude residual maps those 4 numbers to the 3-vector $\delta\theta$, so it
*cannot* see motion along the 4th (the norm/scalar) direction. That direction
therefore gets **zero curvature** in the Gauss–Newton Hessian: $\ell_{xx}$ has
exactly one null eigenvalue. It is harmless — the dynamics keep $\lVert q\rVert=1$,
so the state never actually moves along it, and the solver's damping (§5) keeps
the matrices it inverts non-singular regardless. §4.5 checks this explicitly.

<div align="center"><svg viewBox="0 0 720 210" xmlns="http://www.w3.org/2000/svg" font-family="system-ui,sans-serif" font-size="12.5">
  <rect x="1" y="1" width="718" height="208" rx="10" fill="#ffffff" stroke="#e2e8f0"/>
  <defs><marker id="ah4b" markerWidth="9" markerHeight="9" refX="6" refY="3" orient="auto"><path d="M0,0 L6,3 L0,6 Z" fill="#334155"/></marker></defs>
  <text x="130" y="26" text-anchor="middle" font-weight="600" fill="#0f172a">4 numbers</text>
  <rect x="35" y="36" width="190" height="30" fill="#e0e7ff" stroke="#3730a3"/><text x="130" y="56" text-anchor="middle" fill="#0f172a">q = (qw, qx, qy, qz)</text>
  <text x="130" y="86" text-anchor="middle" fill="#64748b" font-size="11.5">constrained: &#8214;q&#8214; = 1</text>
  <line x1="230" y1="51" x2="300" y2="51" stroke="#334155" stroke-width="1.6" marker-end="url(#ah4b)"/>
  <text x="265" y="42" text-anchor="middle" fill="#64748b" font-size="11">2&#183;vec(q&#8331;&#185;q)</text>
  <text x="415" y="26" text-anchor="middle" font-weight="600" fill="#0f172a">3 real DOF</text>
  <rect x="310" y="36" width="210" height="30" fill="#fde68a" stroke="#b45309"/><text x="415" y="56" text-anchor="middle" fill="#0f172a">&#948;&#952; = (&#948;&#952;x, &#948;&#952;y, &#948;&#952;z)</text>
  <text x="415" y="86" text-anchor="middle" fill="#64748b" font-size="11.5">the actual rotation error</text>
  <line x1="415" y1="98" x2="415" y2="122" stroke="#334155" stroke-width="1.6" marker-end="url(#ah4b)"/>
  <rect x="120" y="126" width="590" height="70" rx="7" fill="#fff7ed" stroke="#c2410c"/>
  <text x="415" y="147" text-anchor="middle" fill="#0f172a" font-size="12.5">the cost sees only these 3 &#8594; the 4th (norm) direction has zero curvature</text>
  <text x="415" y="166" text-anchor="middle" fill="#0f172a" font-size="12.5">&#8594; &#8467;&#8339;&#8339; has exactly one null eigenvalue</text>
  <text x="415" y="185" text-anchor="middle" fill="#0f172a" font-size="12.5">harmless: the rollout keeps &#8214;q&#8214;=1, so the state never moves there</text>
</svg></div>

The per-block **weights** (positive scalars, one per error type) set the relative
importance: position hardest, then attitude, then velocity, body rates lightly,
and a small effort penalty. They enter as $\sqrt{\text{weight}}$ multiplying each
residual entry, so that $\lVert r\rVert^2$ reproduces the usual quadratic
$e^\top Q\,e$. The values below are sensible starting points; §6 tunes them.

In [ ]:
# reference symbols
prx, pry, prz          = sp.symbols('p_rx p_ry p_rz', real=True)
qrw, qrx, qry, qrz     = sp.symbols('q_rw q_rx q_ry q_rz', real=True)
vrx, vry, vrz          = sp.symbols('v_rx v_ry v_rz', real=True)
wrx, wry, wrz          = sp.symbols('w_rx w_ry w_rz', real=True)
ur1, ur2, ur3, ur4     = sp.symbols('u_r1 u_r2 u_r3 u_r4', real=True)

def quat_mul(a, b):
    aw, ax, ay, az = a; bw, bx, by, bz = b
    return sp.Matrix([aw*bw-ax*bx-ay*by-az*bz,
                      aw*bx+ax*bw+ay*bz-az*by,
                      aw*by-ax*bz+ay*bw+az*bx,
                      aw*bz+ax*by-ay*bx+az*bw])
def quat_conj(a): return sp.Matrix([a[0], -a[1], -a[2], -a[3]])

q_cur   = sp.Matrix([q_w, q_x, q_y, q_z])
q_ref   = sp.Matrix([qrw, qrx, qry, qrz])
q_err   = quat_mul(quat_conj(q_ref), q_cur)          # rotation ref -> current
att_err = 2*sp.Matrix([q_err[1], q_err[2], q_err[3]])  # small-angle dtheta

# per-block residual weights = sqrt of the Q/R diagonal (tuned in the closed-loop step)
w_p, w_q, w_v, w_w, w_u = 6.0, 2.0, 1.0, 0.30, 0.10

r_state = sp.Matrix([
    w_p*(r_x-prx), w_p*(r_y-pry), w_p*(r_z-prz),
    w_q*att_err[0], w_q*att_err[1], w_q*att_err[2],
    w_v*(v_x-vrx), w_v*(v_y-vry), w_v*(v_z-vrz),
    w_w*(w_x-wrx), w_w*(w_y-wry), w_w*(w_z-wrz)])
r_ctrl = sp.Matrix([w_u*(T1-ur1), w_u*(T2-ur2), w_u*(T3-ur3), w_u*(T4-ur4)])
print("state residual dim:", r_state.shape[0], " | control residual dim:", r_ctrl.shape[0])

### 4.4 The terminal cost, and why a finite horizon needs one

The sum runs only to $N$: the MPC sees a *finite* window and is blind to
everything after it. If the last thing the cost charges is the stage cost at
$N-1$, the planner has no reason to arrive at $x_N$ in a *recoverable* state — it
will happily end the horizon flung out at high speed, because "after $N$" is free.
Short horizons then track poorly and can destabilise the closed loop.

The **terminal cost** $\ell_f(x_N)$ is the fix: it stands in for *the cost of the
entire flight beyond the horizon*. Ideally it is the infinite-horizon optimal
cost-to-go (a control-Lyapunov function); with it, nominal MPC stability can be
proved (Mayne, Rawlings, Rao & Scokaert, *Constrained model predictive control:
stability and optimality*, Automatica 2000). We do not have that function in
closed form, so — as is standard — we use a **heavier-weighted copy of the state
tracking cost** as a practical surrogate:
$\ell_f(x)=\tfrac12\lVert\sqrt{W_{\text{term}}}\,r_{\text{state}}(x)\rVert^2$.
It is a heuristic, not a certificate (§6.3 shows exactly where that bites); the
weight $W_{\text{term}}$ is the knob that says "really be near the reference by
the end of the window.\"

In [ ]:
J_x = r_state.jacobian(state)      # 12 x 13
J_u = r_ctrl.jacobian(ctrl)        # 4 x 4

l_stage = sp.Rational(1,2)*(r_state.dot(r_state) + r_ctrl.dot(r_ctrl))
l_x  = J_x.T * r_state             # 13
l_u  = J_u.T * r_ctrl              # 4
l_xx = J_x.T * J_x                 # 13 x 13 (Gauss-Newton)
l_uu = J_u.T * J_u                 # 4 x 4  (Gauss-Newton)

ref_syms = (prx,pry,prz, qrw,qrx,qry,qrz, vrx,vry,vrz, wrx,wry,wrz, ur1,ur2,ur3,ur4)
all_syms = (*state, *ctrl, *ref_syms)
L_l   = sp.lambdify(all_syms, l_stage, 'numpy')
L_lx  = sp.lambdify(all_syms, l_x,  'numpy')
L_lu  = sp.lambdify(all_syms, l_u,  'numpy')
L_Jx  = sp.lambdify(all_syms, J_x,  'numpy')
L_Ju  = sp.lambdify(all_syms, J_u,  'numpy')

def _pack(x, u, ref): return (*x, *u, *ref)
def cost(x, u, ref):     return float(L_l(*_pack(x, u, ref)))
def cost_lx(x, u, ref):  return np.asarray(L_lx(*_pack(x, u, ref)), float).flatten()
def cost_lu(x, u, ref):  return np.asarray(L_lu(*_pack(x, u, ref)), float).flatten()
def cost_Jx(x, u, ref):  return np.asarray(L_Jx(*_pack(x, u, ref)), float)
def cost_Ju(x, u, ref):  return np.asarray(L_Ju(*_pack(x, u, ref)), float)

### 4.5 Acid test — analytic derivatives vs finite differences, at many points

A gradient is only trustworthy if it matches a **finite-difference** of the cost
it claims to differentiate. We check $\ell_x,\ell_u$ against a central difference
of $\ell$ **at several random operating points** (a single lucky point could hide
a bug), and confirm the Gauss–Newton Hessians are positive semidefinite: $\ell_{uu}$
strictly (it equals $w_u^2 I$), and $\ell_{xx}$ with the single expected null
direction from §4.3 (its second-smallest eigenvalue is positive).

In [ ]:
def _rand_unit_quat(rng):
    q = rng.normal(size=4); return q/np.linalg.norm(q)

rng = np.random.default_rng(0)
worst_lx = worst_lu = 0.0
worst_min_eig_xx = np.inf; worst_second_eig_xx = np.inf
eps, T_h = 1e-6, m_val*g_val/4.0
for _ in range(12):
    x_op = np.concatenate([rng.normal(0,0.6,3), _rand_unit_quat(rng), rng.normal(0,0.5,6)])
    u_op = T_h + rng.normal(0, 0.8, 4)
    ref  = np.concatenate([rng.normal(0,0.8,3), _rand_unit_quat(rng),
                           rng.normal(0,0.4,3), rng.normal(0,0.3,3), [T_h]*4])
    lx_fd = np.array([(cost(x_op+e, u_op, ref) - cost(x_op-e, u_op, ref))/(2*eps)
                      for e in eps*np.eye(13)])
    lu_fd = np.array([(cost(x_op, u_op+e, ref) - cost(x_op, u_op-e, ref))/(2*eps)
                      for e in eps*np.eye(4)])
    worst_lx = max(worst_lx, np.max(np.abs(cost_lx(x_op,u_op,ref) - lx_fd)))
    worst_lu = max(worst_lu, np.max(np.abs(cost_lu(x_op,u_op,ref) - lu_fd)))
    Jx_n = cost_Jx(x_op,u_op,ref); ev = np.linalg.eigvalsh(Jx_n.T @ Jx_n)
    worst_min_eig_xx    = min(worst_min_eig_xx, ev[0])
    worst_second_eig_xx = min(worst_second_eig_xx, ev[1])

print(f"over 12 random points:")
print(f"  max |l_x - finite diff| = {worst_lx:.2e}")
print(f"  max |l_u - finite diff| = {worst_lu:.2e}")
print(f"  l_xx smallest eigenvalue  = {worst_min_eig_xx:.2e}   (the one null direction, SS4.3)")
print(f"  l_xx 2nd-smallest eigenvalue = {worst_second_eig_xx:.3f}   (> 0: PSD on the real DOF)")
assert worst_lx < 1e-6 and worst_lu < 1e-6
assert worst_min_eig_xx > -1e-9 and worst_second_eig_xx > 1e-6
print("COST ACID TEST PASSED (all points)")

## 5. iLQR / control-limited DDP

We now assemble §2–§4 into the solver. **iLQR** (iterative LQR) is the
Gauss–Newton form of **DDP** (Differential Dynamic Programming); the
**control-limited** variant (Tassa, Erez & Todorov, 2014) adds actuator bounds.
The idea is simple and worth stating plainly: an optimal-control problem over a
whole trajectory is hard, but *around a given guess trajectory* it is well
approximated by a linear–quadratic (LQR-like) problem, which has a closed-form
solution. So we solve that easy problem to get an improved trajectory, and repeat.
Each iteration is three moves (see the diagram):

<div align="center"><svg viewBox="0 0 720 270" xmlns="http://www.w3.org/2000/svg" font-family="system-ui,sans-serif" font-size="12.5">
  <rect x="1" y="1" width="718" height="268" rx="10" fill="#ffffff" stroke="#e2e8f0"/>
  <defs><marker id="ah5" markerWidth="9" markerHeight="9" refX="6" refY="3" orient="auto"><path d="M0,0 L6,3 L0,6 Z" fill="#334155"/></marker></defs>
  <text x="360" y="26" text-anchor="middle" font-weight="600" fill="#0f172a">one iLQR iteration &#8212; repeat until the cost stops dropping</text>
  <rect x="40"  y="60" width="180" height="70" rx="8" fill="#dbeafe" stroke="#1e40af"/>
  <text x="130" y="88" text-anchor="middle" fill="#0f172a" font-weight="600">1 &#183; Forward rollout</text>
  <text x="130" y="108" text-anchor="middle" fill="#334155" font-size="11.5">apply u, run true f</text>
  <text x="130" y="123" text-anchor="middle" fill="#334155" font-size="11.5">&#8594; trajectory + cost J</text>
  <rect x="270" y="60" width="180" height="70" rx="8" fill="#fde68a" stroke="#b45309"/>
  <text x="360" y="84" text-anchor="middle" fill="#0f172a" font-weight="600">2 &#183; Backward pass</text>
  <text x="360" y="103" text-anchor="middle" fill="#334155" font-size="11.5">sweep k = N &#8594; 0, carry</text>
  <text x="360" y="118" text-anchor="middle" fill="#334155" font-size="11.5">cost-to-go V; box-QP &#8594; k&#7584;, K</text>
  <rect x="500" y="60" width="180" height="70" rx="8" fill="#dcfce7" stroke="#15803d"/>
  <text x="590" y="84" text-anchor="middle" fill="#0f172a" font-weight="600">3 &#183; Line search</text>
  <text x="590" y="103" text-anchor="middle" fill="#334155" font-size="11.5">shrink step &#945; until</text>
  <text x="590" y="118" text-anchor="middle" fill="#334155" font-size="11.5">J actually decreases</text>
  <line x1="222" y1="95" x2="268" y2="95" stroke="#334155" stroke-width="1.6" marker-end="url(#ah5)"/>
  <line x1="452" y1="95" x2="498" y2="95" stroke="#334155" stroke-width="1.6" marker-end="url(#ah5)"/>
  <path d="M590,132 L590,170 L130,170 L130,132" fill="none" stroke="#334155" stroke-width="1.6" marker-end="url(#ah5)"/>
  <text x="360" y="188" text-anchor="middle" fill="#334155" font-size="11.5">accepted step &#8594; next iteration</text>
  <rect x="150" y="204" width="420" height="50" rx="7" fill="#fff7ed" stroke="#c2410c"/>
  <text x="360" y="224" text-anchor="middle" fill="#0f172a" font-size="12">damping &#956; (Levenberg): if step 3 fails, raise &#956; &#8594; smaller, safer step;</text>
  <text x="360" y="242" text-anchor="middle" fill="#0f172a" font-size="12">if it succeeds, lower &#956; &#8594; bolder step next time</text>
</svg></div>

1. **Forward rollout** — apply the current command sequence through the *true*
   nonlinear dynamics $F_d$ to get the trajectory it produces and its total cost.
2. **Backward pass** — sweep from the end of the horizon back to the start,
   carrying a local quadratic model of the *cost-to-go* (the cost still to be paid
   from here to the end) and reading off, at each knot, a command correction
   $\delta u_k = k^{ff}_k + K_k\,\delta x_k$ (a feedforward push plus a feedback
   gain). §5.1–5.2 detail this; the actuator bounds enter here.
3. **Line search** — the quadratic model is only local, so we do not trust the full
   correction blindly: we scale it by $\alpha\le 1$ and re-roll, shrinking $\alpha$
   until the cost genuinely drops.

Wrapped around the three is a **damping** parameter $\mu$ (Levenberg–Marquardt
style): when an iteration fails to improve, we raise $\mu$, which biases the
backward pass towards a smaller, safer step; when it succeeds, we lower $\mu$ to
allow bolder steps. This is what lets the method recover gracefully when the local
quadratic model is a poor guide (§5.4).

### 5.1 The backward pass, one knot at a time

At each knot the backward pass builds a quadratic model of the cost-to-go and asks:
*what command change most reduces it?* Writing $V_x,V_{xx}$ for the gradient and
Hessian of the cost-to-go carried back from the next knot, it forms the
**action-value** terms — the cost-to-go expressed as a function of a change in
state and command — by combining the stage cost (§4) with the linearised dynamics
(§3):

$$Q_x=\ell_x+A_k^\top V_x,\qquad Q_u=\ell_u+B_k^\top V_x,$$
$$Q_{xx}=\ell_{xx}+A_k^\top V_{xx}A_k,\qquad
  Q_{uu}=\ell_{uu}+B_k^\top V_{xx}B_k,\qquad
  Q_{ux}=B_k^\top V_{xx}A_k.$$

Read them as: "the cost-to-go changes by $Q_u$ per unit command, curves like
$Q_{uu}$ in the command, and is coupled to the state through $Q_{ux}$." Minimising
this quadratic over $\delta u$ with **no bounds** gives the closed form
$\delta u = -Q_{uu}^{-1}(Q_u + Q_{ux}\,\delta x)$ — i.e. a feedforward
$k^{ff}=-Q_{uu}^{-1}Q_u$ and a feedback gain $K=-Q_{uu}^{-1}Q_{ux}$. That is
**exactly an LQR gain, recomputed along the trajectory** — the "iterated LQR"
name. The Gauss–Newton Hessians of §4 keep $Q_{uu}$ positive definite (with the
damping $\mu$ added), so the inverse is well posed.

In [ ]:
def box_qp(H, g, lo, hi, maxiter=60):
    x = np.clip(np.zeros(len(g)), lo, hi)
    free = np.ones(len(x), bool)
    for _ in range(maxiter):
        grad = g + H @ x
        clamped = ((x <= lo) & (grad > 0)) | ((x >= hi) & (grad < 0))
        free = ~clamped
        if not free.any():
            break
        Hf = H[np.ix_(free, free)]
        gf = (g + H @ (x * clamped))[free]        # hold clamped coords fixed
        try:
            Lc = np.linalg.cholesky(Hf + 1e-9*np.eye(free.sum()))
        except np.linalg.LinAlgError:
            break
        xf = np.linalg.solve(Lc.T, np.linalg.solve(Lc, -gf))
        step = np.zeros(len(x)); step[free] = xf - x[free]
        if np.max(np.abs(step)) < 1e-10:
            break
        a, c0 = 1.0, 0.5*x@H@x + g@x               # projected backtracking
        for _ in range(20):
            xn = np.clip(x + a*step, lo, hi)
            if 0.5*xn@H@xn + g@xn <= c0:
                x = xn; break
            a *= 0.5
        else:
            x = np.clip(x + a*step, lo, hi)
    return x, free

### 5.2 Actuator limits — the box-QP

Real motors saturate, so $\delta u$ must keep each command inside
$[T_\mathrm{min},T_\mathrm{max}]$. Instead of the unconstrained inverse above, the
feedforward comes from a small **box-constrained quadratic program**:

$$\min_{\delta u}\; \tfrac12\,\delta u^\top Q_{uu}\,\delta u + Q_u^\top\delta u
  \quad\text{s.t.}\quad T_\mathrm{min}-u_k \le \delta u \le T_\mathrm{max}-u_k.$$

We solve it with **projected Newton** (Tassa et al., 2014): repeatedly split the
motors into *clamped* (sitting at a bound with the gradient pushing further out —
freeze them) and *free* (the rest), take a Newton step on the free block via a
Cholesky factorisation of its $Q_{uu}$ sub-matrix, and project back into the box
with a short line search. It returns the minimiser **and the free set** — and the
feedback gain $K$ is allowed to act only on the free motors (a clamped motor cannot
respond, so its feedback row is zero). This is the "control-limited" in
control-limited DDP: the bound is respected *inside* the plan, not clipped after.

In [ ]:
T_MIN, T_MAX = 0.0, 36.0     # per-rotor thrust box [N] (runtime PhysicsParams)
W_TERM       = 8.0           # terminal state-weight multiplier

def ref_at(refs, k):
    # a single 17-vector is broadcast over the whole horizon; a per-stage
    # list/2D-array (length N+1) is indexed -> lets the MPC track a trajectory.
    return refs if (isinstance(refs, np.ndarray) and refs.ndim == 1) else refs[k]

def backward_pass(xs, us, refs, mu):
    N = len(us)
    rN = ref_at(refs, N)
    Jx  = cost_Jx(xs[N], u_ref, rN)
    Vx  = W_TERM * cost_lx(xs[N], u_ref, rN)
    Vxx = W_TERM * (Jx.T @ Jx)
    kff = [None]*N; Kg = [None]*N
    for k in range(N-1, -1, -1):
        rk = ref_at(refs, k)
        A, B = rk4_jacobians(xs[k], us[k])
        lx = cost_lx(xs[k], us[k], rk); lu = cost_lu(xs[k], us[k], rk)
        Jxk = cost_Jx(xs[k], us[k], rk); Juk = cost_Ju(xs[k], us[k], rk)
        lxx = Jxk.T @ Jxk; luu = Juk.T @ Juk
        Qx  = lx + A.T @ Vx
        Qu  = lu + B.T @ Vx
        Qxx = lxx + A.T @ Vxx @ A
        Quu = luu + B.T @ Vxx @ B + mu*np.eye(4)
        Qux = B.T @ Vxx @ A
        k_i, free = box_qp(Quu, Qu, T_MIN - us[k], T_MAX - us[k])
        K_i = np.zeros((4, 13))
        if free.any():
            K_i[free, :] = -np.linalg.solve(Quu[np.ix_(free, free)], Qux[free, :])
        kff[k], Kg[k] = k_i, K_i
        Vx  = Qx + K_i.T @ Quu @ k_i + K_i.T @ Qu + Qux.T @ k_i
        Vxx = Qxx + K_i.T @ Quu @ K_i + K_i.T @ Qux + Qux.T @ K_i
        Vxx = 0.5*(Vxx + Vxx.T)
    return kff, Kg

### 5.3 Forward pass and line search

The backward pass proposes a correction; the forward pass *tries* it. It applies
$\delta u_k=\alpha\,k^{ff}_k+K_k(x'_k-x_k)$ through the **true** nonlinear map $F_d$
(so the new trajectory is dynamically exact, not a linear prediction), clamps each
command to the box, and accepts the first step size $\alpha\in\{1,\tfrac12,\dots\}$
that lowers the total cost. If none does, the iteration is rejected and the damping
$\mu$ is raised. Because both the rollout and the clamp use the real, bounded
dynamics, **every accepted iterate is feasible** — dynamically and in the actuators.

In [ ]:
def make_ref(p, q, v, w, u):        # pack the 17-vector the section 4 cost expects
    return np.concatenate([p, q, v, w, u]).astype(float)

def rollout(x0, us):
    xs = [x0]
    for k in range(len(us)):
        xs.append(F_d(xs[k], us[k]))
    return xs

def traj_cost(xs, us, refs):
    J = sum(cost(xs[k], us[k], ref_at(refs, k)) for k in range(len(us)))
    return J + W_TERM * cost(xs[-1], u_ref, ref_at(refs, len(us)))   # control residual = 0 at u_ref

def forward_pass(xs, us, kff, Kg, refs, J0):
    for a in (1.0, 0.5, 0.25, 0.125, 0.0625, 0.03, 0.015, 0.007, 0.0):
        xn = [xs[0]]; un = []
        for k in range(len(us)):
            du = a*kff[k] + Kg[k] @ (xn[k] - xs[k])
            uk = np.clip(us[k] + du, T_MIN, T_MAX); un.append(uk)
            xn.append(F_d(xn[k], uk))
        Jn = traj_cost(xn, un, refs)
        if Jn < J0:
            return xn, un, Jn, a
    return xs, us, J0, 0.0

### 5.4 The solver loop — putting it together

The loop is: roll out, backward pass, line search; on success shrink $\mu$ and keep
the new trajectory, on failure grow $\mu$ and retry; stop when the relative cost
improvement falls below a tolerance (converged) or $\mu$ blows up (stuck). The
damping $\mu$ is the safety valve of §5's opening: near the solution the quadratic
model is trustworthy, $\mu$ stays small and steps are near-Newton (fast); far from
it, a failed step pushes $\mu$ up and the next step is short and conservative.

In [ ]:
def ilqr(x0, refs, N, iters=60, tol=1e-6, us_init=None):
    us = [u.copy() for u in us_init] if us_init is not None else [u_ref.copy() for _ in range(N)]
    xs = rollout(x0, us)
    J  = traj_cost(xs, us, refs)
    mu = 1e-3; hist = [J]
    for _ in range(iters):
        kff, Kg = backward_pass(xs, us, refs, mu)
        xs2, us2, J2, a = forward_pass(xs, us, kff, Kg, refs, J)
        if a > 0:
            rel = (J - J2) / max(J, 1e-9)
            xs, us, J = xs2, us2, J2
            mu = max(mu*0.7, 1e-6); hist.append(J)
            if rel < tol:
                break
        else:
            mu *= 4.0; hist.append(J)
            if mu > 1e3:
                break
    return xs, us, hist

### 5.5 Demo — stabilise from a perturbed hover

Start 1.8 m off target with a tilt and a roll rate, reference the hover setpoint,
and let the receding-horizon problem ($N=60$, $\Delta t=20$ ms $\Rightarrow$ 1.2 s
preview) drive it home. The cost decreases monotonically and the four motor
commands stay inside the box.

In [ ]:
import matplotlib.pyplot as plt

u_ref = np.array([m_val*g_val/4.0]*4)                         # hover thrust per rotor
ref   = make_ref([0,0,0], [1,0,0,0], [0,0,0], [0,0,0], u_ref)

x0 = normalize_quat(np.array([1.5,-1.0,0.5, 1,0.10,0.05,0, 0,0,0, 0.2,0,0], float))
xs, us, hist = ilqr(x0, ref, N=60, iters=60)
U = np.array(us); X = np.array(xs)

print(f"cost {hist[0]:.1f} -> {hist[-1]:.1f} in {len(hist)-1} iterations")
print(f"final position error = {np.linalg.norm(X[-1,0:3]):.4f} m")
print(f"controls span [{U.min():.3f}, {U.max():.3f}] N   (box [{T_MIN}, {T_MAX}])")
assert U.min() >= T_MIN-1e-9 and U.max() <= T_MAX+1e-9
assert all(hist[i+1] <= hist[i]+1e-9 for i in range(len(hist)-1)), "cost not monotone"

t = np.arange(len(us))*DT_MPC
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].semilogy(hist, '-o', ms=3); ax[0].set_title("cost per iteration"); ax[0].set_xlabel("iter"); ax[0].grid(alpha=.3)
for i in range(3): ax[1].plot(t, X[:-1, i], label="xyz"[i])
ax[1].set_title("position [m]"); ax[1].set_xlabel("t [s]"); ax[1].legend(); ax[1].grid(alpha=.3)
for i in range(4): ax[2].plot(t, U[:, i], label=f"T{i+1}")
ax[2].axhline(T_MAX, ls='--', c='k', lw=.8); ax[2].axhline(T_MIN, ls='--', c='k', lw=.8)
ax[2].set_title("motor thrusts [N]"); ax[2].set_xlabel("t [s]"); ax[2].legend(ncol=2, fontsize=8); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

### 5.6 Constraints that bite — tighten the actuator bound

To see the control-limited part earn its name, re-solve an aggressive far-start
with a deliberately tight upper thrust. Several motors ride the bound for a stretch
— the QP keeps every command feasible while the still-free motors do the steering,
and the solve still converges.

In [ ]:
T_MAX_saved = T_MAX
T_MAX = 9.0                                  # tight bound to force saturation
x0b = normalize_quat(np.array([4.0,0,0, 1,0,0,0, 0,0,0, 0,0,0], float))
xsb, usb, histb = ilqr(x0b, ref, N=80, iters=80)
Ub = np.array(usb)
n_sat = int(np.sum((Ub >= T_MAX-1e-6) | (Ub <= T_MIN+1e-6)))
print(f"cost {histb[0]:.1f} -> {histb[-1]:.1f}, saturated samples {n_sat}/{Ub.size}, "
      f"final pos err {np.linalg.norm(np.array(xsb)[-1,0:3]):.4f} m")
assert Ub.min() >= T_MIN-1e-9 and Ub.max() <= T_MAX+1e-9, "box violated"

tb = np.arange(len(usb))*DT_MPC
plt.figure(figsize=(6.2, 3.2))
for i in range(4): plt.plot(tb, Ub[:, i], label=f"T{i+1}")
plt.axhline(T_MAX, ls='--', c='k', lw=.8); plt.axhline(T_MIN, ls='--', c='k', lw=.8)
plt.title(f"motor thrusts vs a tight bound (T_max={T_MAX} N)")
plt.xlabel("t [s]"); plt.ylabel("N"); plt.legend(ncol=2, fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()
T_MAX = T_MAX_saved                          # restore the physical bound

### 5.7 Tracking an aggressive trajectory into hard saturation

The solver now takes a **per-stage reference** `refs[k]`, so we can hand it a real
trajectory instead of a single setpoint. Here the reference makes a fast
6 m + 4 m lateral and 3 m climb concentrated in the **middle third** of the horizon
(a `smootherstep`), and we deliberately tighten the thrust bound to
$T_\mathrm{max}=10$ N — barely above the $mg/4\approx5.9$ N hover thrust.

The demanded acceleration peaks mid-flight, so the motors slam into the bound
right there: the box-QP keeps every command feasible, the vehicle **cuts the
corner** because it physically cannot follow, and the plots show exactly where
and how the saturation happens — motor commands, the tracked 3-D path, and the
per-axis position with the saturated window shaded.

In [ ]:
def smootherstep(a):     # 6a^5 - 15a^4 + 10a^3 (C2, flat ends)
    u = np.clip(a, 0.0, 1.0); return u**3*(10 - 15*u + 6*u**2)
def smootherstep_d(a):
    u = np.clip(a, 0.0, 1.0); return np.where((a >= 0) & (a <= 1), 30*u**2*(1 - u)**2, 0.0)

N_tr  = 120                                  # 2.4 s horizon at 50 Hz
t_tr  = np.arange(N_tr + 1)*DT_MPC
T_end = N_tr*DT_MPC
f0, f1 = 0.35, 0.65                          # transition = middle third of the horizon
phase  = (t_tr/T_end - f0)/(f1 - f0)
s      = smootherstep(phase)
s_d    = smootherstep_d(phase)/((f1 - f0)*T_end)

P0 = np.array([0.0, 0.0, 0.0]); P1 = np.array([6.0, 4.0, 3.0])
p_ref = P0 + np.outer(s,   P1 - P0)          # (N_tr+1, 3) position reference
v_ref = np.outer(s_d, P1 - P0)               # its exact derivative (velocity reference)
refs  = [make_ref(p_ref[k], [1,0,0,0], v_ref[k], [0,0,0], u_ref) for k in range(N_tr + 1)]

T_MAX_saved = T_MAX
T_MAX = 10.0                                 # tight bound -> hard mid-trajectory saturation
x0 = normalize_quat(np.array([0,0,0, 1,0,0,0, 0,0,0, 0,0,0], float))
xs, us, hist = ilqr(x0, refs, N=N_tr, iters=120)
X = np.array(xs); U = np.array(us)

sat = (U >= T_MAX-1e-6) | (U <= T_MIN+1e-6)  # (N_tr, 4) boolean saturation mask
n_sat = sat.sum(axis=1)                       # motors at a bound per step
track_err = np.linalg.norm(X[:, 0:3] - p_ref, axis=1)
print(f"cost {hist[0]:.0f} -> {hist[-1]:.0f} in {len(hist)-1} iters")
print(f"peak simultaneous saturated motors: {n_sat.max()} / 4")
print(f"steps with >=1 motor saturated: {int((n_sat>0).sum())} / {N_tr}")
print(f"max tracking error: {track_err.max():.3f} m   (corner is cut under saturation)")
assert U.min() >= T_MIN-1e-9 and U.max() <= T_MAX+1e-9, "box violated"
T_MAX = T_MAX_saved

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
sat_any = n_sat > 0
t_u = t_tr[:-1]
win = t_u[sat_any]
span = (win.min(), win.max()) if sat_any.any() else None

fig = plt.figure(figsize=(13, 4.2))
# (a) 3-D tracked path
ax0 = fig.add_subplot(1, 3, 1, projection='3d')
ax0.plot(p_ref[:,0], p_ref[:,1], p_ref[:,2], 'k--', lw=1.3, label='reference')
ax0.plot(X[:,0], X[:,1], X[:,2], 'C0', lw=1.8, label='actual')
sat_pts = X[:-1][sat_any]
if len(sat_pts): ax0.scatter(sat_pts[:,0], sat_pts[:,1], sat_pts[:,2], c='r', s=12, label='saturated')
ax0.set_title('tracked trajectory'); ax0.set_xlabel('x'); ax0.set_ylabel('y'); ax0.set_zlabel('z')
ax0.legend(fontsize=8)
# (b) motor thrusts
ax1 = fig.add_subplot(1, 3, 2)
for i in range(4): ax1.plot(t_u, U[:,i], label=f'T{i+1}')
ax1.axhline(T_MAX, ls='--', c='k', lw=.8); ax1.axhline(T_MIN, ls='--', c='k', lw=.8)
ax1.axhline(m_val*g_val/4, ls=':', c='gray', lw=.9)
if span: ax1.axvspan(*span, color='r', alpha=.10)
ax1.set_title('motor thrusts [N]'); ax1.set_xlabel('t [s]'); ax1.legend(ncol=2, fontsize=8); ax1.grid(alpha=.3)
# (c) position tracking + saturation count
ax2 = fig.add_subplot(1, 3, 3)
for i,cl in enumerate('xyz'):
    ax2.plot(t_tr, p_ref[:,i], '--', c=f'C{i}', lw=1); ax2.plot(t_tr, X[:,i], c=f'C{i}', lw=1.6, label=cl)
if span: ax2.axvspan(*span, color='r', alpha=.10)
ax2b = ax2.twinx(); ax2b.fill_between(t_u, n_sat, step='mid', color='r', alpha=.15)
ax2b.set_ylabel('# motors saturated', color='r'); ax2b.set_ylim(0, 4.3)
ax2.set_title('position: ref (--) vs actual'); ax2.set_xlabel('t [s]'); ax2.legend(fontsize=8); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 6. Closed loop — receding horizon

Everything so far solved **one** open-loop problem from a fixed start. A real MPC
runs *receding horizon*: each control tick it measures the state, solves the
horizon problem, applies **only the first command** $u_0$, slides the window one
step, and repeats. Two properties make that worthwhile:

- **Feedback by re-planning.** Re-solving from the *measured* state rejects
  disturbances and model error the original plan never saw.
- **Warm starting.** The previous solution, shifted one step ($u_{1:N}$ then a
  repeat of the last command), is an excellent initial guess, so a few iLQR
  iterations replace a cold solve.

To show what "re-planning" actually buys — beyond merely *having* feedback — we
race three controllers built from the **same** trajectory optimisation, on the
same reference and the same gust:

| | applies | rejects disturbance? | re-plans? |
|---|---|---|---|
| **(a) feedforward** | the planned $u_t$ only | no | no |
| **(b) trajectory-LQR** | $u_t + K_t(x-x^{\text{plan}}_t)$ from one solve | yes (small) | no |
| **(c) MPC** | re-solves every tick from $x$ | yes | yes |

(b) is the honest baseline: it is exactly the tracking controller you get from a
*single* iLQR solve — feedforward plus the solver's own time-varying feedback
gains $K_t$. Comparing (c) against (b) isolates the value of *re-optimising*
(not of feedback) — and part of the lesson is how small that margin is when the
disturbance is smooth.

### 6.1 Reference, gust, and the four controllers

A feasible fast dash (2.5 m + 1.5 m lateral, 1 m climb) so all three track well in
nominal conditions; a lateral velocity **gust** kicks the plant mid-run. `F_d`
advances the true plant; the gust is injected as a state kick the planners never
model. `plan_once` produces the nominal $x^{\text{plan}}_t,u_t$ and the feedback
gains $K_t$ that (a) and (b) reuse without re-solving.

In [ ]:
SIM, N_h = 70, 25
tot = SIM + N_h + 1
tg  = np.arange(tot)*DT_MPC; T_move = SIM*DT_MPC
ph  = (tg/T_move - 0.15)/(0.80 - 0.15)
spr = smootherstep(ph); spd = smootherstep_d(ph)/((0.80-0.15)*T_move)
Pa, Pb = np.array([0,0,0.]), np.array([2.5, 1.5, 1.0])
p_ref = Pa + np.outer(spr, Pb - Pa)
v_ref = np.outer(spd, Pb - Pa)
ref_traj = np.array([make_ref(p_ref[k], [1,0,0,0], v_ref[k], [0,0,0], u_ref) for k in range(tot)])

G0, G1 = 32, 44                                          # gust window (ticks)
def gust(t):
    g = np.zeros(13)
    if G0 <= t < G1: g[8] = -0.18                        # lateral velocity kick [m/s per step]
    return g
x0 = normalize_quat(np.array([0,0,0, 1,0,0,0, 0,0,0, 0,0,0], float))

def plan_once():                                         # one full-horizon solve -> plan + gains
    xs, us, _ = ilqr(x0, ref_traj[:SIM+1], SIM, iters=80)
    _, Kg = backward_pass(xs, us, ref_traj[:SIM+1], 1e-6)
    return xs, us, Kg

def sim_apply(policy):                                   # roll the true plant under a feedback policy
    x = x0.copy(); X = [x]
    for t in range(SIM):
        u = np.clip(policy(t, x), T_MIN, T_MAX)
        x = normalize_quat(F_d(x, u) + gust(t)); X.append(x)
    return np.array(X)

def mpc_run(sim_steps=SIM, warm=True):                   # (c) receding horizon
    x = x0.copy(); X = [x]; iters = []; us = [u_ref.copy() for _ in range(N_h)]
    for t in range(sim_steps):
        _, us_sol, hist = ilqr(x, ref_traj[t:t+N_h+1], N_h,
                               iters=8 if warm else 60, us_init=us if warm else None)
        iters.append(len(hist)-1)
        x = normalize_quat(F_d(x, np.clip(us_sol[0], T_MIN, T_MAX)) + gust(t)); X.append(x)
        us = us_sol[1:] + [us_sol[-1]]
    return np.array(X), np.array(iters)

In [ ]:
xs_p, us_p, Kg_p = plan_once()
Xa = sim_apply(lambda t, x: us_p[t])                              # (a) feedforward
Xb = sim_apply(lambda t, x: us_p[t] + Kg_p[t] @ (x - xs_p[t]))    # (b) trajectory-LQR
Xc, it_c = mpc_run()                                             # (c) MPC (warm)
_,  it_cold = mpc_run(sim_steps=15, warm=False)                  # cold, short, for the iter count

def track_err(X): return np.linalg.norm(X[:SIM+1, 0:3] - p_ref[:SIM+1], axis=1)
ea, eb, ec = track_err(Xa), track_err(Xb), track_err(Xc)
for name, e in [("(a) feedforward ", ea), ("(b) trajectory-LQR", eb), ("(c) MPC         ", ec)]:
    print(f"{name}: RMS {np.sqrt((e**2).mean()):.3f} m, peak {e.max():.3f} m, post-gust settle {e[-1]:.3f} m")
print(f"warm start: {it_c.mean():.1f} iters/tick (peak {it_c.max()})  vs  cold {it_cold.mean():.1f}")

In [ ]:
tu = np.arange(SIM)*DT_MPC; tf = np.arange(SIM+1)*DT_MPC
fig = plt.figure(figsize=(13, 4.2))
ax0 = fig.add_subplot(1, 3, 1, projection='3d')
ax0.plot(p_ref[:SIM+1,0], p_ref[:SIM+1,1], p_ref[:SIM+1,2], 'k--', lw=1.2, label='reference')
ax0.plot(Xc[:,0], Xc[:,1], Xc[:,2], 'C0', lw=1.8, label='(c) MPC')
ax0.plot(Xb[:,0], Xb[:,1], Xb[:,2], 'C1', lw=1.3, label='(b) traj-LQR')
ax0.plot(Xa[:,0], Xa[:,1], Xa[:,2], 'C3', lw=1.3, label='(a) feedforward')
ax0.set_title('closed-loop trajectory'); ax0.set_xlabel('x'); ax0.set_ylabel('y'); ax0.set_zlabel('z'); ax0.legend(fontsize=8)
ax1 = fig.add_subplot(1, 3, 2)
ax1.plot(tf, ea, 'C3', label='(a) feedforward'); ax1.plot(tf, eb, 'C1', label='(b) traj-LQR'); ax1.plot(tf, ec, 'C0', lw=1.8, label='(c) MPC')
ax1.axvspan(G0*DT_MPC, G1*DT_MPC, color='gray', alpha=.15, label='gust')
ax1.set_title('position tracking error [m]'); ax1.set_xlabel('t [s]'); ax1.legend(fontsize=8); ax1.grid(alpha=.3)
ax2 = fig.add_subplot(1, 3, 3)
ax2.plot(tu, it_c, 'C2', drawstyle='steps-mid'); ax2.axhline(it_cold.mean(), ls='--', c='gray', lw=.9, label=f'cold avg {it_cold.mean():.1f}')
ax2.set_title('MPC iterations per tick (warm)'); ax2.set_xlabel('t [s]'); ax2.set_ylim(bottom=0); ax2.legend(fontsize=8); ax2.grid(alpha=.3)
plt.tight_layout(); plt.show()

The gust hits all three about equally (similar peak); what separates them is what
happens next. **Feedback is decisive**: pure feedforward (a) never corrects and
drifts away, settling more than a metre off, while both feedback controllers stay
within a few tens of centimetres. Between (b) and (c) the result is closer and more
honest than a slogan — MPC rides the nonlinear excursion with a slightly lower peak
and RMS by re-planning each tick, while the trajectory-LQR, whose gains come from a
*full-horizon* solve, settles marginally tighter. The real takeaway: for a smooth
disturbance a well-built trajectory-LQR rivals MPC; MPC's structural edge is
previewing an upcoming actuator limit and re-optimising when a disturbance changes
*which* constraints bind — the regime of §5.6-5.7 — not a single mild gust.
Meanwhile warm starting holds the re-solve to a few iterations per tick, the
practical line between running in real time and not.

### 6.3 Leaving the tube — where re-planning is decisive

The gust above kept the state in a *tube* around the nominal, so (b) and (c)
tracked almost identically: a frozen feedback gain is enough there. The honest
place MPC pulls decisively ahead is when the state leaves that tube.

Recall what (b) actually is: its gains $K_t$ are linearised about the **nominal**
$x^{\text{plan}}_t$, so the correction $K_t(x-x^{\text{plan}}_t)$ is a *first-order*
model — valid only for small $x-x^{\text{plan}}_t$. Here we make the nominal a plain
hover (so $K_t$ is essentially the constant hover gain — the same controller as the
FF+LQR notebook) and start the vehicle **far from it**: rolled past vertical, with a
body rate and a sideways velocity. Below ~90° both recover. Past ~90° the
linearisation is simply wrong: (b) commands a correction computed about upright and
**flies away**, while (c) re-linearises and re-optimises from the *actual* attitude
and rights itself.

This is also the honest answer to "is the MPC guaranteed to stabilise?" — **no
certificate**. It recovers across the envelope below, but the residual left at
120° versus the clean recovery at 160° is the fingerprint of a *local* optimiser
with a heuristic terminal cost and a short horizon: it works here, it is not
proven to.

In [ ]:
SIM_s, N_s = 70, 25
ref_hover = make_ref([0,0,0], [1,0,0,0], [0,0,0], [0,0,0], u_ref)
ref_stab  = np.tile(ref_hover, (SIM_s + N_s + 1, 1))
x_hover   = normalize_quat(np.array([0,0,0, 1,0,0,0, 0,0,0, 0,0,0], float))

# (b) TVLQR: gains sewn onto the hover nominal -> ~ the constant hover gain of the FF+LQR notebook
xs_nom, us_nom, _ = ilqr(x_hover, ref_stab[:SIM_s+1], SIM_s, iters=60)
_, Kg_nom = backward_pass(xs_nom, us_nom, ref_stab[:SIM_s+1], 1e-6)

def q_roll(phi):  return [np.cos(phi/2), np.sin(phi/2), 0.0, 0.0]
def att_deg(x):   return np.degrees(2*np.arccos(np.clip(abs(x[3]), 0.0, 1.0)))

def tvlqr_from(x0):                                   # (b) frozen gains, no re-solve
    x = x0.copy(); X = [x]
    for t in range(SIM_s):
        du = Kg_nom[t] @ (x - xs_nom[t])
        x = normalize_quat(F_d(x, np.clip(us_nom[t] + du, T_MIN, T_MAX))); X.append(x)
    return np.array(X)

def mpc_from(x0):                                     # (c) re-solve each tick from the true state
    x = x0.copy(); X = [x]; us = [u_ref.copy() for _ in range(N_s)]
    for t in range(SIM_s):
        _, us_sol, _ = ilqr(x, ref_stab[t:t+N_s+1], N_s, iters=8, us_init=us)
        x = normalize_quat(F_d(x, np.clip(us_sol[0], T_MIN, T_MAX))); X.append(x)
        us = us_sol[1:] + [us_sol[-1]]
    return np.array(X)

In [ ]:
runs = {}
print("initial roll        TVLQR  final(att / dist)        MPC  final(att / dist)")
for phi_deg, wx, vy in [(45,1.0,-0.8), (80,2.0,-1.5), (120,2.5,-1.8), (160,3.0,-2.0)]:
    x0 = normalize_quat(np.array([0,0,0, *q_roll(np.radians(phi_deg)), 0,vy,0, wx,0,0], float))
    Xb, Xc = tvlqr_from(x0), mpc_from(x0)
    runs[phi_deg] = (Xb, Xc)
    print(f"  {phi_deg:3d} deg        {att_deg(Xb[-1]):5.1f} deg / {np.linalg.norm(Xb[-1,0:3]):6.2f} m"
          f"          {att_deg(Xc[-1]):5.1f} deg / {np.linalg.norm(Xc[-1,0:3]):6.2f} m")

In [ ]:
Xb, Xc = runs[120]; th = np.arange(SIM_s+1)*DT_MPC
fig = plt.figure(figsize=(13, 3.8))
ax0 = fig.add_subplot(1, 3, 1)
ax0.plot(th, [att_deg(x) for x in Xb], 'C1', label='(b) TVLQR — frozen gains')
ax0.plot(th, [att_deg(x) for x in Xc], 'C0', lw=1.8, label='(c) MPC — re-plan')
ax0.axhline(90, ls=':', c='r', lw=.8, label='90° (tube edge)')
ax0.set_title('attitude error [deg], 120° start'); ax0.set_xlabel('t [s]'); ax0.legend(fontsize=8); ax0.grid(alpha=.3)
ax1 = fig.add_subplot(1, 3, 2)
ax1.plot(th, np.linalg.norm(Xb[:,0:3], axis=1), 'C1'); ax1.plot(th, np.linalg.norm(Xc[:,0:3], axis=1), 'C0', lw=1.8)
ax1.set_title('distance from hover [m]'); ax1.set_xlabel('t [s]'); ax1.grid(alpha=.3)
ax2 = fig.add_subplot(1, 3, 3, projection='3d')
ax2.plot(Xb[:,0], Xb[:,1], Xb[:,2], 'C1', label='(b) TVLQR')
ax2.plot(Xc[:,0], Xc[:,1], Xc[:,2], 'C0', lw=1.8, label='(c) MPC')
ax2.scatter([0],[0],[0], c='k', s=25, label='hover target')
ax2.set_title('trajectory: fly-away vs recovery'); ax2.set_xlabel('x'); ax2.set_ylabel('y'); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 7. C++ export — the prediction model for the runtime

The runtime does not need a *control law* from us: the MPC controller is a
hand-written C++ solver (`box_qp`, backward/forward pass) that lives outside the
generated code. What it *does* need — and all it needs — from this notebook is the
**prediction model**: the discrete map is built at run time from the continuous
dynamics, so we emit exactly

- `Dynamics(s, u, userF)` — the continuous $f(x,u)$ with the external world force, and
- `Jacobians(s, u) → f_x, f_u` — from which the solver assembles $A_k,B_k$ (§3).

The generator `mpc_codegen.py` mirrors the FF+LQR one but emits no `ExecuteControl`
and no gain literal. Its output under `exported_cpp/QUADROTOR_MPC_01/` is
**generated code — never edit it by hand**; change the model here and re-run.

In [ ]:
import mpc_codegen as mcg
import subprocess, os, tempfile

T_max_s, T_min_s = sp.symbols('T_max_s T_min_s', positive=True)   # box bounds: solver-side, absent from f

cfg = mcg.mpc_config()
gen = (mcg.MpcCodegen(cfg)
       .set_state_symbols(state).set_input_symbols(ctrl)
       .set_physics_symbols([m, Ixx, Iyy, Izz, g, d_x, d_y, d_z, k_T, k_Q, L, T_max_s, T_min_s])
       .set_user_force_symbols([uFx, uFy, uFz])
       .set_dynamics(f).set_jacobians(fx, fu))
hpath, cpath = gen.write()
print("emitted:\n ", hpath, "\n ", cpath)

### 7.1 Regression test — compile the C++ and compare to the notebook

Compile the emitted model with a tiny driver, evaluate `Dynamics` and `Jacobians`
at a random non-trivial point (with a non-zero external force), and check they
match `f_num`, `fx_num`, `fu_num` to machine precision — the same acid test as
§3.1, now across the Python↔C++ boundary.

In [ ]:
rng = np.random.default_rng(7)
xs_t = rng.normal(0, 0.4, 13); xs_t[3:7] /= np.linalg.norm(xs_t[3:7])
us_t = m_val*g_val/4.0 + rng.normal(0, 1.0, 4)
uF_t = np.array([0.7, -1.1, 0.3])

f_ref  = f_num(xs_t, us_t, uF_t)
fx_ref = fx_num(xs_t, us_t)
fu_ref = fu_num(xs_t, us_t)

driver = rf'''#include "{os.path.abspath(hpath)}"
#include <cstdio>
using M = CDS::Dynamics::{cfg.model_name};
int main() {{
  M mdl;
  M::StateVec   s  = {{{",".join(repr(float(v)) for v in xs_t)}}};
  M::InputVec   u  = {{{",".join(repr(float(v)) for v in us_t)}}};
  M::UserForces uF = {{{",".join(repr(float(v)) for v in uF_t)}}};
  auto dx = mdl.Dynamics(s, u, uF);
  for (double v : dx) printf("%.15g ", v); printf("\n");
  double fx[13][13], fu[13][4]; mdl.Jacobians(s, u, fx, fu);
  for (int i=0;i<13;i++) for(int j=0;j<13;j++) printf("%.15g ", fx[i][j]); printf("\n");
  for (int i=0;i<13;i++) for(int j=0;j<4;j++)  printf("%.15g ", fu[i][j]); printf("\n");
  return 0;
}}'''
d = tempfile.mkdtemp(); dp = os.path.join(d, "drv.cpp"); open(dp, "w").write(driver)
subprocess.run(["clang++","-std=c++20","-O2", dp, os.path.abspath(cpath), "-o", os.path.join(d,"drv")], check=True)
o = subprocess.run([os.path.join(d,"drv")], capture_output=True, text=True).stdout.split("\n")
f_c  = np.array(o[0].split(), float)
fx_c = np.array(o[1].split(), float).reshape(13, 13)
fu_c = np.array(o[2].split(), float).reshape(13, 4)
print("max|f  C++ - notebook| =", f"{np.max(np.abs(f_c  - f_ref)):.2e}")
print("max|fx C++ - notebook| =", f"{np.max(np.abs(fx_c - fx_ref)):.2e}")
print("max|fu C++ - notebook| =", f"{np.max(np.abs(fu_c - fu_ref)):.2e}")
assert max(np.max(np.abs(f_c-f_ref)), np.max(np.abs(fx_c-fx_ref)), np.max(np.abs(fu_c-fu_ref))) < 1e-9
print("REGRESSION OK — the exported C++ matches the notebook math.")

## 8. What we have, and what's left

The notebook is complete end to end: model, cost, solver, closed loop, and now the
**C++ prediction model** exported and regression-tested against this notebook.

- **prediction** $F_d,\;A_k,B_k$ (§2–§3), **cost** with Gauss–Newton Hessians (§4),
- **iLQR / control-limited DDP** with a box-QP and line search (§5), a
  **receding-horizon** loop with warm start and disturbance rejection (§6), and
- the exported model `QUADROTOR_MPC_01` (`Dynamics`, `Jacobians`) — §7.

**Left for the runtime (hand-written C++, outside this notebook):**

- port `box_qp`, `backward_pass`, `forward_pass` into an MPC controller that owns
  the model and the warm-start trajectory across ticks, samples the reference over
  the horizon via `GetReference`, and applies $u_0$ as ZOH — the stateful
  counterpart of the LQR's stateless `ExecuteControl`.